# putEMG — Deep Learning Cross-Subject Models (LOSO)

Trains and evaluates deep learning architectures on the putEMG gesture dataset using
**Leave-One-Subject-Out (LOSO)** cross-validation. Set `MODEL_TYPE` in the config cell to switch models.

| Model | Description |
|-------|-------------|
| **EMG_TCN** | Spatial mixing + 4 dilated temporal conv blocks |
| **EEGNet** | Depthwise separable CNN |
| **ShallowConvNet** | Temporal + spatial conv, square/log nonlinearity |

**LOSO split (per fold):**
- **Test** — 1 held-out subject (never seen during training)
- **Val** — 10% of the remaining 43 subjects, stratified by class (early stopping only)
- **Train** — remaining 90% of those 43 subjects

Weights are saved per model under `weights/<MODEL_TYPE>/`; per-fold logs under `results/results_log_<MODEL_TYPE>.txt`.
Completed folds are skipped automatically — safe to stop and resume at any time.

> **Prerequisites**: Run `data_preprocessing/driver.ipynb` first to generate per-subject `.mat` files.

In [ ]:
import os
import sys
import numpy as np
import torch

In [ ]:
sys.path.append(os.path.abspath('../../../'))

import src.deep_learning_models as dlm
from src.emg_loader import load_all_subjects
from src.loso_trainer import train, evaluate, evaluateFinal, run_loso, update_log

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
_notebook_dir = os.path.abspath(os.getcwd())

# -- Paths --
DATA_DIR    = '/Volumes/KRIS/data/UG_per_subject'
MODEL_TYPE  = 'EMG_TCN'   # change to: 'EEGNet' or 'ShallowConvNet'
WEIGHTS_DIR = os.path.join(_notebook_dir, 'weights', MODEL_TYPE)
RESULTS_DIR = os.path.join(_notebook_dir, 'results')
LOG_PATH    = os.path.join(RESULTS_DIR, f'results_log_{MODEL_TYPE}.txt')

# -- Data split --
BATCH_SIZE = 16
VAL_FRAC   = 0.10

# -- Training --
MAX_EPOCHS = 20
PATIENCE   = 5
MIN_DELTA  = 0.002
LR         = 1e-3
DROPOUT    = 0.1

MODEL_MAP = {
    'EMG_TCN':        dlm.EMG_TCN,
    'EEGNet':         dlm.EEGNet,
    'ShallowConvNet': dlm.ShallowConvNet,
}

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
subjects = load_all_subjects(DATA_DIR)
print(f'Loaded {len(subjects)} subjects')
print(f'Weights → {WEIGHTS_DIR}')
print(f'Log     → {LOG_PATH}')

---
## LOSO Training Loop

Trains the selected model (`MODEL_TYPE`) on every subject as a test fold:

- **Test** — 1 held-out subject, never seen during training
- **Train / Val** — all other 43 subjects, split 90/10 (stratified by class, seeded) within the pool

Checkpointing:
- Weights saved to `weights/<MODEL_TYPE>/<MODEL_TYPE>_<subject_id>.pt` after each fold
- If a checkpoint already exists the fold is skipped — safe to stop and resume at any time
- `results_log_<MODEL_TYPE>.txt` is rebuilt from all checkpoints after every fold

In [ ]:
run_loso(
    subjects    = subjects,
    model_cls   = MODEL_MAP[MODEL_TYPE],
    model_type  = MODEL_TYPE,
    weights_dir = WEIGHTS_DIR,
    log_path    = LOG_PATH,
    device      = device,
    val_frac    = VAL_FRAC,
    batch_size  = BATCH_SIZE,
    dropout     = DROPOUT,
    lr          = LR,
    max_epochs  = MAX_EPOCHS,
    patience    = PATIENCE,
    min_delta   = MIN_DELTA,
)

---
## Results

In [ ]:
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        print(f.read())
else:
    print('No results yet — run the training loop first.')